# Day 1: Prompt Engineering and Structured Output

week 5, day 1. this whole week is about actually building with LLMs instead of just understanding how they work, so today's about how you talk to one. using `google/flan-t5-base` here instead of Day 6's base GPT-2, on purpose, it's instruction-tuned so it can actually follow a task instead of just continuing text. that matters because every comparison in this notebook only means something if the model is capable of following instructions at all in the first place.

In [1]:
import ssl
import certifi
ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())
##Using flan t5
from transformers import T5Tokenizer, T5ForConditionalGeneration

tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")

print("flan-t5-base loaded.")

# quick note: unlike week 4 Day 6's base GPT-2, this model is instruction-tuned,
#  A plain instruction should actually get followed instead of just continued
inputs = tokenizer("Summarize: The stock market rallied today after the Fed signaled a possible rate cut next quarter.", return_tensors="pt")
output = model.generate(**inputs, max_new_tokens=30)
print(tokenizer.decode(output[0], skip_special_tokens=True))
##the output demo'd hallucination, it made up information about Dow Jones closing up 0.16% at 17,016, which is not anywhere

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 4588.87it/s]


[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


flan-t5-base loaded.


(Close): The Dow Jones closed up 0.16% at 17,016.


flan-t5 actually followed the instruction this time instead of looping like Day 6's GPT-2 did. but look closer at what it said, it invented a specific Dow Jones number and closing price that isn't anywhere in the input. that's hallucination, and a genuinely good example of it, the model didn't fail to follow instructions, it followed them confidently while making up a fact. keeping this one for later, it comes back up.

In [2]:
##prep for prompting
def ask(prompt, max_new_tokens=30):
    inputs = tokenizer(prompt, return_tensors="pt")
    output = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(output[0], skip_special_tokens=True)

## zero shot vs few shot (multinomial) classification
##the same 6 categories used all through project: Tech, Markets, Business, Politics, Health, Energy
headline = "Tesla stock jumps after new self-driving software update"

zero_shot_prompt = f"""Classify the news headline into one category: Technology, Markets, Business, Politics, Health, or Energy.) 
Headline: {headline}
Category:"""

##few shot prompt
few_shot_prompt = f"""Classify the news headline into one category: Technology, Markets, Business, Politics, Health, or Energy.) 
Headline: Senate approves new tarrifs on imported steel
Category: Politics

Headline: Oil prices climb as OPEC signals production cuts
Category: Energy

Headline: FDA approves new cancer drug after fast-tracked trial
Category: Health

Headline: {headline}
Category:"""

##output
print("\n--- Zero shot classification---")
print("Prompt:", zero_shot_prompt)
print("Output:", ask(zero_shot_prompt, max_new_tokens=6))

print("\n--- Few shot classification---")
print("Prompt:", few_shot_prompt)
print("Output:", ask(few_shot_prompt, max_new_tokens=6))
##both prompts output 'Tech' but the headline was a giveaway too, try with something more vague
## ORIGINAL TESTED HEADLINE: headline = "Nvidia unveils new AI chip aimed at data center customers"

##new prompt output business rather than committing to either markets or technology, noting for later
##few shot did not help improve the answer, as any vague categories have been getting absorbed into the 'business' category


--- Zero shot classification---
Prompt: Classify the news headline into one category: Technology, Markets, Business, Politics, Health, or Energy.) 
Headline: Tesla stock jumps after new self-driving software update
Category:
Output: Business

--- Few shot classification---
Prompt: Classify the news headline into one category: Technology, Markets, Business, Politics, Health, or Energy.) 
Headline: Senate approves new tarrifs on imported steel
Category: Politics

Headline: Oil prices climb as OPEC signals production cuts
Category: Energy

Headline: FDA approves new cancer drug after fast-tracked trial
Category: Health

Headline: Tesla stock jumps after new self-driving software update
Category:
Output: Business


first test headline (Nvidia AI chip) was too easy, both zero-shot and few-shot nailed "Technology" instantly, which doesn't actually prove anything about whether the examples helped. swapped to a genuinely ambiguous one instead, a Tesla stock move caused by a tech product, the same kind of Markets/Technology overlap Day 5's NER work already ran into on this exact dataset.

both zero-shot and few-shot landed on "Business" here instead of committing to either Markets or Technology. the few-shot examples didn't sharpen the answer at all, the model just defaulted to the generic catch-all category on anything ambiguous, same overlap Week 2 Day 4's clustering work found: Business absorbs whatever doesn't cleanly fit anywhere else.

In [3]:
##------Role prompting: summarization------
article_text = "Nvidia announced a new AI chip today aimed at data center customers, saying it will boost training speeds by 30% over the previous generation. Shares rose 4% in after-hours trading following the announcement."

plain_prompt = f"Summarize this one one sentence: {article_text}"

role_prompt = f"""You are a financial news editor wiriting a one-sentence headline summary for busy investors. Be concise and lead with the most important fact in the headline.
Article{article_text}
Summary:"""

print("\n--- Plain summarization ---")
print("Output:", ask(plain_prompt, max_new_tokens=40))

print("\n--- Role-prompted summarization ---")
print("Output:", ask(role_prompt, max_new_tokens=40))
##output worked well, plain summarization did what it is supposed to: Output: Nvidia's AI chip is expected to boost data center training speeds.
##Role prompted: Output: Nvidia announces new AI chip for data center customers
##this was a shift in style based on the given persona in the role prompting, a good output result


--- Plain summarization ---


Output: Nvidia's AI chip is expected to boost data center training speeds.

--- Role-prompted summarization ---
Output: Nvidia announces new AI chip for data center customers


role prompt did shift the style, terser, more headline-like ("announces" vs "is expected to"). but here's the catch: neither summary actually mentions the 4% after-hours stock move, even though the role explicitly said to lead with what matters most to investors. that's arguably the most investor-relevant fact in the whole article. so role prompting changed tone/format here but not what the model actually judged important, a real limit worth remembering, not something to chase away with more instruction tweaking.

In [4]:
##-----prompt templates for structured output / extraction-----
headline_for_extraction = "Apple stock rose 3% after the company reported record iPhone sales in its Q4 earnings call."

unstructured_prompt = f"Extract the company name and the stock price change from this headline: {headline_for_extraction}"

structured_prompt = f"""Extract information from the headline below and respond in exactly this format:
Company: <name>
Change: <percentage>

Headline: {headline_for_extraction}"""

print("\n--- Unstructured extraction ---")
print("Output:", ask(unstructured_prompt, max_new_tokens=30))

print("\n--- Structured (templated) extraction ---")
print("Output:", ask(structured_prompt, max_new_tokens=30))
## both outputs failed in the task
##unstructured did give back 'Apple' but dropped any percentage
##structured responed in a summary format and did not follow the format guidline it was guided to
##re-testing with new set
structured_prompt_v2 = f"""Extract information from the headline and respond in exactly this format.

Headline: Tesla stock fell 5% after missing delivery targets.
Company: Tesla
Change: -5%

Headline: {headline_for_extraction}
Company:"""

print("\n--- Structured extraction, with a worked example ---")
print("Output:", ask(structured_prompt_v2, max_new_tokens=30))
##same failure pattern observed, model is not able to respond in a format


--- Unstructured extraction ---


Output: Apple

--- Structured (templated) extraction ---


Output: Apple shares rose 3% after the company reported record iPhone sales in its Q4 earnings call.

--- Structured extraction, with a worked example ---


Output: Apple shares rose 3% after the company reported record iPhone sales in its Q4 earnings call.


this is the most honest negative result in the whole notebook. the unstructured prompt only pulled "Apple" and dropped the percentage entirely, only grabbed one of the two things asked for. the structured prompt just described the format and ignored it completely, responded with a plain summary sentence instead of `Company: / Change:`. tried again giving it one worked example instead of just describing the format (few-shot + template combined), and it still just echoed the input sentence back.

real finding: structured field extraction is genuinely hard for a model this size, and it's not a prompt-wording problem, showing an example didn't fix it either. prompting alone can't reliably force format compliance here. this is exactly the kind of thing fine-tuning and RAG exist to solve with actual control over the pipeline instead of hoping the wording of one prompt gets it right.

In [5]:
##-----temperature, top-k, top-p (nucleus sampling)-----
##do_sample = False always picks the single most likely token (greedy)
##do_sample = True introduces controlled randomness instead
creative_prompt = "Write a one-sentence headline about a tech company's earnings report."

def ask_sampling(prompt, temperature=1.0, top_k=50, top_p=1.0, max_new_tokens=30):
    inputs = tokenizer(prompt, return_tensors="pt")
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

print("\n--- Greedy (deterministic) ---")
print("Output:", ask(creative_prompt, max_new_tokens=30))

print("\n--- Low temperature (0.3), sampling ---")
print("Output:", ask_sampling(creative_prompt, temperature=0.3))

print("\n--- High temperature (1.5), sampling ---")
print("Output:", ask_sampling(creative_prompt, temperature=1.5))

print("\n--- top_p=0.5 (nucleus sampling, narrow) ---")
print("Output:", ask_sampling(creative_prompt, top_p=0.5))
##ran twice, high-temp and top_p had more different and creative examples each time
##greedy gave the same result back each time
##low-temp matched greedy but was slightly different
##temp- 1.5 was a nonsense answer
##top_p was the only one that made some sense on both runs


--- Greedy (deterministic) ---
Output: Tech giant reports earnings

--- Low temperature (0.3), sampling ---
Output: Tech giant reports earnings

--- High temperature (1.5), sampling ---


Output: Wallecom's Profit, But Waller Has Been Very Closer

--- top_p=0.5 (nucleus sampling, narrow) ---
Output: Tech giant EPS reports


ran this twice to actually see the randomness in action, not just read about it.

greedy came back identical both times, "Tech giant reports earnings", that's what fully deterministic looks like. temperature 0.3 mostly hugged the greedy answer (matched it exactly on the second run) but not always, low temperature narrows randomness toward the safest choice, it doesn't remove it completely. temperature 1.5 gave two totally different outputs across runs, and one of them ("Isolation & the IPO #OddThingsUp%20B") was genuinely incoherent, a real example of turning randomness up too far breaking the output instead of just making it more "creative". top_p=0.5 stayed readable on both runs while still varying, more controlled variety than raw high temperature.

**temperature** scales how sharply the model favors its top guesses. **top_k** only considers the k most likely next tokens no matter what. **top_p** keeps adding tokens until their combined probability crosses p, adapting to how confident the model is in the moment instead of using a fixed count.

In [6]:
## ------ Q&A: zero-shot vs. context-grounded ------
question = "What caused Nvidia's stock to rise?"

no_context_prompt = f"Question: {question}\nAnswer:"

context = "Nvidia announced a new AI chip today aimed at data center customers, saying it will boost training speeds by 30% over the previous generation. Shares rose 4% in after-hours trading following the announcement."

grounded_prompt = f"""Answer the question using only the context above. If the answer isn't in the context, say "not enough information."

Context: {context}
Question: {question}
Answer:"""

print("\n--- Q&A with no context ---")
print("Output:", ask(no_context_prompt, max_new_tokens=30))

print("\n--- Q&A grounded in context ---")
print("Output:", ask(grounded_prompt, max_new_tokens=30))


--- Q&A with no context ---
Output: a computer chip chip

--- Q&A grounded in context ---


Output: a new AI chip


cleanest before/after in the whole notebook. no context, the model has nothing real to pull from and gives back "a computer chip chip", garbled, vague, stutters on itself. with the context handed to it directly, it correctly answers "a new AI chip", pulled straight from the given text instead of guessing.

this is the entire idea behind RAG: give the model real source text to answer from instead of relying on whatever it half-remembers from pretraining. Days 4 and 5 later this week build that for real, retrieval, vector databases, and an actual RAG chatbot, this Q&A test is basically a tiny preview of the exact same idea.

## takeaway

four task types, two prompting strategies each, and the pattern that showed up over and over: prompting can change *how* a model says something (tone, format attempt, style) far more reliably than it can change *what* the model actually gets right. few-shot didn't rescue an ambiguous classification. role prompting shifted style but not priority. describing a structured format, even showing an example of it, still didn't get the model to comply. temperature swings showed exactly where controlled randomness turns into noise. and the one place prompting cleanly solved a real problem was Q&A with context handed directly to the model, which is exactly why RAG exists as the next real step, not just another prompting trick.

also worth remembering: the very first output in this notebook was a hallucinated stock number that was never in the input. that's not a one-off, it's the default behavior of a model with no grounding, confidently making things up. worth keeping in mind for every one of these outputs, structured or not.